In [15]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import base64
import time
import json
import webbrowser
from flask import Flask, request
import os
import shutil
# we imported our libraries
# we will use base64 to decode the data
# its a common way to encode binary data as text
# we used flask to create a web server so we can handle the redirect from spotify
# we will use requests to make http requests
# we will use os and shutil to handle file operations
# os to interact with the operating system
# shutil to handle high-level file operations like copying and removing directories

In [16]:
client_id = "your_client_id"
client_secret = "your_client_secret"
# we set our client id and secret


In [17]:
auth_str = f"{client_id}:{client_secret}"
b64_auth_str = base64.b64encode(auth_str.encode()).decode()

In [18]:
token_url = "https://accounts.spotify.com/api/token"
headers = {"Authorization": f"Basic {b64_auth_str}"}
data = {"grant_type": "client_credentials"}
response = requests.post(token_url, headers=headers, data=data)
access_token = response.json()

In [ ]:
# Spotify credentials (we enter our client id and secret to access the Spotify API)
client_id = 'your_client_id'
client_secret = 'your_client_secret'
redirect_uri = 'http://127.0.0.1:8888/callback'
scope = 'playlist-read-private playlist-read-collaborative'

# Step 1: Direct user to authorize
auth_url = (
    'https://accounts.spotify.com/authorize'
    f'?client_id={client_id}'
    '&response_type=code'
    f'&redirect_uri={redirect_uri}'
    f'&scope={scope.replace(" ", "%20")}'
)
print("Go to the following URL to authorize:")
print(auth_url)
webbrowser.open(auth_url)

# Step 2: Set up Flask server to catch the redirect
app = Flask(__name__)
auth_code = None

@app.route('/callback')
def callback():
    global auth_code
    auth_code = request.args.get('code')
    return "Authorization code received! You can close this tab."

def run_flask():
    app.run(port=8888)

import threading
threading.Thread(target=run_flask).start()

# Wait for user to authorize and get the code
while auth_code is None:
    time.sleep(1)

# Step 3: Exchange code for access token
token_url = 'https://accounts.spotify.com/api/token'
auth_header = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
headers = {
    'Authorization': f'Basic {auth_header}',
    'Content-Type': 'application/x-www-form-urlencoded'
}
data = {
    'grant_type': 'authorization_code',
    'code': auth_code,
    'redirect_uri': redirect_uri
}
response = requests.post(token_url, headers=headers, data=data)
tokens = response.json()
print(tokens)

access_token = tokens['access_token']

# Step 4: Use the access token for API requests
headers = {'Authorization': f'Bearer {access_token}'}
resp = requests.get('https://api.spotify.com/v1/me', headers=headers)
print(resp.json())

In [6]:
# we used the authorization code flow to get an access token
# this allows us to access user-specific data from Spotify API
# by default the api will only give us public data

In [20]:
# List of Spotify album and playlist links
# we are going to extract the IDs from these links
links = [

"https://open.spotify.com/album/1pzvBxYgT6OVwJLtHkrdQK?si=zhZ1c9fMRZa3N12uDSxKGA",

"https://open.spotify.com/album/6DEjYFkNZh67HP7R9PSZvv?si=uauyUVWzQLGo5r2RxZbYGQ",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evNZVVBPG?si=a6d98f1c476b4595",

"https://open.spotify.com/playlist/37i9dQZF1E4sa1kOvRGgMb?si=81355d6ff58b4a88",

"https://open.spotify.com/playlist/37i9dQZF1E4sa1kOvRGgMb?si=81355d6ff58b4a88",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO4dpvkA?si=651a1a36aa784dd0",

"https://open.spotify.com/playlist/37i9dQZF1E4xstu1WxmJS4?si=eb25ad732b2541ad",

"https://open.spotify.com/playlist/37i9dQZF1DX1PfYnYcpw8w?si=c2cd1ad245674a55",

"https://open.spotify.com/playlist/37i9dQZF1DX2i24iHGhL24?si=c521885c7f56450c",

"https://open.spotify.com/playlist/37i9dQZF1E4AfEUiirXPyP?si=35a9643af12b4278",

"https://open.spotify.com/playlist/37i9dQZF1DX5KpP2LN299J?si=78a46ccc6eb044ff",

"https://open.spotify.com/playlist/4T2jzNwcjV7RwcllulrVpp?si=f2d738ad26bf403e",

"https://open.spotify.com/playlist/6PVHgTRAz4bWZTHxtfxTMI?si=b20fd9f103c84d46",

"https://open.spotify.com/playlist/1YxI6tntse0iyEoLxyh7vd?si=c7885c0a6df84b77",

"https://open.spotify.com/playlist/2fmTTbBkXi8pewbUvG3CeZ?si=aa3eac86c31c4009",

"https://open.spotify.com/playlist/2mdnbXPiAYIDZZlQmahyTQ?si=e2b354dc741a47c6",

"https://open.spotify.com/playlist/5l771HfZDqlBsDFQzO0431?si=04fef1a10ac9458f",

"https://open.spotify.com/playlist/0zrFhK5EuAQcQkrrY9yQqt?si=7c5226bdef7f40ac",

"https://open.spotify.com/playlist/0BCcDQPimJfkKkzLb5IzBP?si=fcac36fc284b49d6",

"https://open.spotify.com/playlist/0zAHPBsGRrEH4p2CPbtsLK?si=ec53b875e02641e2",

"https://open.spotify.com/playlist/5XALIurWS8TuF6kk8bj438?si=4bb0ff7dd27843f4",

"https://open.spotify.com/playlist/37i9dQZF1DWVRSukIED0e9?si=2f45efc57e8749e8",

"https://open.spotify.com/playlist/0PMKjSoU937cvzkHpFJ3hf?si=9bf56c6ae70b43f9",

"https://open.spotify.com/playlist/414ImOZ0R7hRgmA7DX17bb?si=5017ed83df3e4bd0",

"https://open.spotify.com/playlist/774kUuKDzLa8ieaSmi8IfS?si=720ff791f3f04c39",

"https://open.spotify.com/playlist/6unJBM7ZGitZYFJKkO0e4P?si=cc0db22ad1bf4055",

"https://open.spotify.com/playlist/5KJDMJe9EJ7QRz8FG2MIpI?si=903e3025f49a4427",

"https://open.spotify.com/playlist/37i9dQZEVXbNG2KDcFcKOF?si=54fe9da9ba7142bd",

"https://open.spotify.com/playlist/2mzlU8dfi5qqdWahPvbQyE?si=303f97f21916449b",

"https://open.spotify.com/playlist/24DKKckhcPEXACdqlYTvdh?si=180b0c5722f349fe",

"https://open.spotify.com/playlist/34NbomaTu7YuOYnky8nLXL?si=21e7bade210c4cd5",

"https://open.spotify.com/playlist/37i9dQZF1DX50MDbDwt4w8?si=f3b19ad4214d4a41",

"https://open.spotify.com/playlist/3uEIq4MtL84iIVDoZIFVBh?si=5947f7f2741f41d1",

"https://open.spotify.com/playlist/7E3uEa1emOcbZJuB8sFXeK?si=f72f4ad8993e448e",

"https://open.spotify.com/playlist/37i9dQZEVXbLn7RQmT5Xv2?si=2558ab9feac6407b",

"https://open.spotify.com/playlist/37i9dQZF1E37PlRIlu7ZAX?si=1c54b4e3cdb74681",

"https://open.spotify.com/playlist/37i9dQZF1E37jU98gyLoYL?si=f00199bb067b4941",

"https://open.spotify.com/playlist/37i9dQZEVXcR0Ip4HFcvyq?si=9b7c0181c8134e1d",

"https://open.spotify.com/playlist/37i9dQZF1E35UhJHxObSv5?si=367ba033c14d441f",

"https://open.spotify.com/playlist/37i9dQZF1E36bdElP5dSnB?si=88408cd737eb4356",

"https://open.spotify.com/playlist/37i9dQZF1E39DoZi5s6tVy?si=615d398fbd504364",

"https://open.spotify.com/playlist/37i9dQZF1E39d1NBRtY3GS?si=8aab57d2b5ba464b",

"https://open.spotify.com/playlist/37i9dQZF1DX1lVhptIYRda?si=8d7bb95ef61a453c",

"https://open.spotify.com/playlist/37i9dQZF1DXbS5WTN5nKF7?si=90398f99f08047ac",

"https://open.spotify.com/playlist/37i9dQZF1DXcRXFNfZr7Tp?si=9cce3162f71b4e85",

"https://open.spotify.com/playlist/37i9dQZF1DX4IDaXtVjL83?si=51ae4d0b837d4e74",

"https://open.spotify.com/playlist/37i9dQZF1DXcBWIGoYBM5M?si=96ff2eee2d844c07",

"https://open.spotify.com/playlist/37i9dQZF1DWYs83FtTMQFw?si=7643776bf78e4916",

"https://open.spotify.com/playlist/37i9dQZF1EQnqst5TRi17F?si=dc4f2c1ac82a419e",

"https://open.spotify.com/playlist/3PMsbsTsmBhuuESfYCOdY1?si=ed3b930e8ae945f1",

"https://open.spotify.com/playlist/37i9dQZF1EIdZFdTlGR1gX?si=58967327e8234553",

"https://open.spotify.com/playlist/37i9dQZF1EIfeeY1Nyg89M?si=456fb2dd077a41c3",

"https://open.spotify.com/playlist/37i9dQZF1EQoqCH7BwIYb7?si=e89df1a4c68f4643",

"https://open.spotify.com/playlist/37i9dQZF1DWY3PJWG3ogmJ?si=e55b57a4cf054ef9",

"https://open.spotify.com/playlist/37i9dQZF1DX08jcQJXDnEQ?si=b41e8fc1b39e4802",

"https://open.spotify.com/playlist/6P0p1x7RK3g2lJyhE4mFu6?si=99033ac95bf2487f",

"https://open.spotify.com/playlist/37i9dQZF1DWXRqgorJj26U?si=38a95c5cfacd42da",

"https://open.spotify.com/playlist/7DgPQwzEoUVfQYBiMLER9Z?si=c7a56d741ad04298",

"https://open.spotify.com/playlist/37i9dQZF1EQpj7X7UK8OOF?si=0b53f101c7704f0c",

"https://open.spotify.com/playlist/7fIdjWhWzPbf1g5OSDKOXw?si=aa0429fe613b4064",

"https://open.spotify.com/playlist/7fIdjWhWzPbf1g5OSDKOXw?si=aa0429fe613b4064",

"https://open.spotify.com/playlist/7iUA1FAtsZoKgUXxtjZ0eH?si=e27c9b65bae8490b",

"https://open.spotify.com/playlist/1WH6WVBwPBz35ZbWsgCpgr?si=fcddf3e3ed314476",

"https://open.spotify.com/playlist/37i9dQZF1EQncLwOalG3K7?si=61706028de324360",

"https://open.spotify.com/playlist/4WsU99qurCxQn1bFZewGab?si=d90d38359c794e29",

"https://open.spotify.com/playlist/5ABHKGoOzxkaa28ttQV9sE?si=fae2bbfa906149ec",

"https://open.spotify.com/playlist/37i9dQZEVXbMDoHDwVN2tF?si=1053cb408abf4570",
    
"https://open.spotify.com/playlist/6UeSakyzhiEt4NB3UAd6NQ?si=c04a89617a9a47a3",

"https://open.spotify.com/playlist/45QYqTOzpWUPm4TPcSBqqN?si=b07fda735ecf464f",

"https://open.spotify.com/playlist/37i9dQZF1DXbYM3nMM0oPk?si=d7a76669a997489c",

"https://open.spotify.com/playlist/65uAjFTt4N8sEJeonhNOBL?si=b14bbe840c7e4c61",

"https://open.spotify.com/playlist/3wPe8LCiQ3zeFs9REe6R2k?si=f27dad8112744460",

"https://open.spotify.com/playlist/37i9dQZF1DX2M1RktxUUHG?si=d89de69d6c294b40",

"https://open.spotify.com/album/5L5evi5tJPh8WaEFAQp7Tp?si=jkJsl2r7TlqBIv_rcs-wrA",

"https://open.spotify.com/playlist/1YpRzoYQXOdzKjipNhszyQ?si=6b66a7ceda6e4ef1",

"https://open.spotify.com/playlist/4bx5c78CAquCWNE4tw1reY?si=347bfdfae88f4aa4",

"https://open.spotify.com/playlist/6yYA6aUGp8qUTgQWWYkPkP?si=b106c9ea23bf4705",

"https://open.spotify.com/playlist/0NLeHTjb6gGTMepz5iBxJ3?si=ea1399bc77184684",

"https://open.spotify.com/album/1yGbNOtRIgdIiGHOEBaZWf?si=k0lGq2HPQLGySrxJaX8hzQ",

"https://open.spotify.com/playlist/37i9dQZF1DX2L0iB23Enbq?si=21293bf4aedc43e4",

"https://open.spotify.com/playlist/37i9dQZF1DX0018ciYu6bM?si=d4b3bd068b484e06", 

"https://open.spotify.com/playlist/37i9dQZF1DX3R7OWWGN4gH?si=04110e4a7c98434a",

"https://open.spotify.com/playlist/37i9dQZF1DX2sUQwD7tbmL?si=6c32bbf1fc714f04",

"https://open.spotify.com/playlist/37i9dQZF1DWSThc8QnzIme?si=4ef3165688c74c86",

"https://open.spotify.com/playlist/37i9dQZF1DWSf2RDTDayIx?si=18f2ffadd4b74652",

"https://open.spotify.com/playlist/37i9dQZF1DWUZMtnnlvJ9p?si=4ea7b9db6d9a403c",

"https://open.spotify.com/playlist/37i9dQZF1DWWGzo2lhvYlP?si=2f68a358554b4b4d",

"https://open.spotify.com/playlist/37i9dQZF1DX64iwDddhmfW?si=6224cda571ef471c",

"https://open.spotify.com/playlist/37i9dQZF1DX8S9gwdi7dev?si=c66eedb890a04d97",

"https://open.spotify.com/playlist/37i9dQZF1DX8S9gwdi7dev?si=c66eedb890a04d97",

"https://open.spotify.com/playlist/37i9dQZF1DX0Yxoavh5qJV?si=62dc5b7b400d4652",

"https://open.spotify.com/playlist/3vDe8D64ytZRKXt0AsJT0B?si=4e4e65a9f15c4c5f"

"https://open.spotify.com/playlist/37i9dQZF1DWTyiBJ6yEqeu?si=ba271b8f61004988",

"https://open.spotify.com/playlist/37i9dQZF1Fa6HnFnBxzWRk?si=c6327cd2e6af4bef",

"https://open.spotify.com/playlist/37i9dQZF1F0sijgNaJdgit?si=da5bd1b0c4c14e60",

"https://open.spotify.com/playlist/37i9dQZF1EUMDoJuT8yJsl?si=e06e53e6a7d94dbd",

"https://open.spotify.com/playlist/2pLOvSQPJwCFXqhlk60eeL?si=22cc1cc0b853497a",

"https://open.spotify.com/playlist/2jKZQAXjtpLDVAPcw1jPVP?si=8f14cf7b5fe843c5",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO09rhnz?si=e42cc9b001a7432a",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO09rhnz?si=e42cc9b001a7432a",

"https://open.spotify.com/playlist/37i9dQZEVXbLRQDuF5jeBp?si=fe558520ca28422f",

"https://open.spotify.com/album/2lIZef4lzdvZkiiCzvPKj7?si=x_hah96bQA6NYeKzEETEog",

"https://open.spotify.com/playlist/37i9dQZF1DWZUozJiHy44Y?si=95b7b0a35da643d0",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO2nSXaa?si=acfc284ddbf0476e",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO2uG3nl?si=0bc9fe27051a4140",

"https://open.spotify.com/playlist/37i9dQZF1DX6cg4h2PoN9y?si=ceeda7ce1a374ee2",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3by276?si=e0f6af9618114cde",

"https://open.spotify.com/playlist/37i9dQZF1DX7QOv5kjbU68?si=2dd81ace3b0747e7",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO4gTUOY?si=5748034d71bf4c1c",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO1IPOOk?si=a0bb503312204b04",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3nMr04?si=6301bac2bc43407b",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO42EWMo?si=726af06941a54e88",

"https://open.spotify.com/playlist/37i9dQZF1DXcISkz62UgzG?si=e3364b15c2474d94",

"https://open.spotify.com/playlist/37i9dQZF1DX55yuR78Invt?si=7ed744f13f834dd2",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO2VxlyE?si=06da78359f6f47ac",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO07zaak?si=9ef7a0673e5046c3",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO2iBPiw?si=084ec4994be24575",

"https://open.spotify.com/album/2XGEyGU76kj55OdHWynX0S?si=01BKNX7hRn687hlAUf0NJg",

"https://open.spotify.com/album/0P3oVJBFOv3TDXlYRhGL7s?si=VQdg7ujgQoyBbkawN9UW2g",

"https://open.spotify.com/album/5VoeRuTrGhTbKelUfwymwu?si=4OSdLdThT3WfltD7dbNB6g",

"https://open.spotify.com/album/1ORxRsK3MrSLvh7VQTF01F?si=7w2t0uLTR_CUAs268rR2Hw",

"https://open.spotify.com/playlist/37i9dQZF1DX5Ejj0EkURtP?si=b7bf21d6edf64b1e",

"https://open.spotify.com/album/1KVKqWeRuXsJDLTW0VuD29?si=oxIfrjmgRAqYzVg3jkLQCA",

"https://open.spotify.com/album/6AORtDjduMM3bupSWzbTSG?si=mmPSSsrHQriVuf1F46Xdnw",

"https://open.spotify.com/album/5EYKrEDnKhhcNxGedaRQeK?si=l6YWwrQgSbWhsYiDh-N9gA",

"https://open.spotify.com/album/7aJuG4TFXa2hmE4z1yxc3n?si=sG5ljFlBTheeSeeIFMjnzg",

"https://open.spotify.com/album/0S0KGZnfBGSIssfF54WSJh?si=co8oP092TS-mMzw0wl8Mmw",

"https://open.spotify.com/album/5H7ixXZfsNMGbIE5OBSpcb?si=VpwX-X_gT4a_Ce7Cv01rVQ",

"https://open.spotify.com/playlist/0mQbGXAlmsI3T2Vyfa0IVd?si=31081576b3dc480a",

"https://open.spotify.com/album/1aqg30bNvLSWgShZgX4oop?si=jVSzb9wgSKmzwTgPegGmcQ",

"https://open.spotify.com/album/3iPSVi54hsacKKl1xIR2eH?si=mw-0rPM2Q52dnCv9hRqH-A",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO1aBeik?si=6fe2b58278014f31",

"https://open.spotify.com/playlist/37i9dQZF1DX76Wlfdnj7AP?si=a0ce847b1f74418f",

"https://open.spotify.com/album/4BbsHmXEghoPPevQjPnHXx?si=i7wqU13QStGal1j2t5kCdw",

"https://open.spotify.com/album/5s0rmjP8XOPhP6HhqOhuyC?si=9xpfJ_sjTYeABwisBtYgjw",

"https://open.spotify.com/album/4g1ZRSobMefqF6nelkgibi?si=6_pvU9aSRS-hkSZYsPXAXg",

"https://open.spotify.com/album/1xn54DMo2qIqBuMqHtUsFd?si=NRQ3bWH0Tfyz0aG7ZwbJBA",

"https://open.spotify.com/album/3T4tUhGYeRNVUGevb0wThu?si=-HWIcUESS5GJmj4K9hNLGg",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO33svt5?si=96667ac50b574d22",

"https://open.spotify.com/album/1nAQbHeOWTfQzbOoFrvndW?si=jfjGrYLoRJC2gC2p0OVxUg",

"https://open.spotify.com/album/2cWBwpqMsDJC1ZUwz813lo?si=1n1uLX5XRLepFDWRPSO5hw",

"https://open.spotify.com/album/47BiFcV59TQi2s9SkBo2pb?si=xLJc6iTYSTao_va6qkaHkg",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO2xmY3T?si=15a9baa4d315408c",

"https://open.spotify.com/album/4KdtEKjY3Gi0mKiSdy96ML?si=nEq_JIwNReKclH4URBM4tA",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO0BEOzm?si=12128e7141b7478b",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3wzrW0?si=e7dfd1bbf8f640f8",

"https://open.spotify.com/album/3RDqXDc1bAETps54MSSOW0?si=h07TyUtQSX6SRc7Hr-2OkA",

"https://open.spotify.com/album/5DvJgsMLbaR1HmAI6VhfcQ?si=RzoAir58StmeEytX-X_Iqg",

"https://open.spotify.com/playlist/37i9dQZF1DX3fRquEp6m8D?si=63d1067d849f428e",

"https://open.spotify.com/album/01sfgrNbnnPUEyz6GZYlt9?si=uDI8JtDhS52AYxzbmI3mAg",

"https://open.spotify.com/album/5lKlFlReHOLShQKyRv6AL9?si=eEZmsrZFTHm6GMaULn-BWw",

"https://open.spotify.com/playlist/37i9dQZF1DWTwnEm1IYyoj?si=2676b9de349b4645",

"https://open.spotify.com/album/6TVfiWmo8KtflUAmkK9gGF?si=j7IP3Ys3Tz2y4GeUL1KtiA",

"https://open.spotify.com/album/21jF5jlMtzo94wbxmJ18aa?si=nd_pt9GCRFSAfeRqTBenAw",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO03DwPK?si=ba13bbb307e44580",

"https://open.spotify.com/album/4PgleR09JVnm3zY1fW3XBA?si=9UPSX9D5R8CHO9jBoFdr1g",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3vuU6c?si=deb495546a894f0e",

"https://open.spotify.com/album/3ARwSvDQv2OHYnLeDC3Lxi?si=j3M9CmVuTPOKEaqyd0P3dQ",

"https://open.spotify.com/playlist/37i9dQZF1DXc7FZ2VBjaeT?si=fbb8f4d244e1470b",

"https://open.spotify.com/album/6rePArBMb5nLWEaY9aQqL4?si=6kc_oWFPSaCrlu9PeYKV5w",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3Jefw4?si=81548e716657468a",

"https://open.spotify.com/album/06SY6Ke6mXzZHhURLVU57R?si=KxiOaCULR722V39r0lIP3A",

"https://open.spotify.com/album/4hDok0OAJd57SGIT8xuWJH?si=QT_LTTZWTGq3vTwFe2PXNg",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO0aBNIs?si=06a2466f8da048d0",

"https://open.spotify.com/album/2fYhqwDWXjbpjaIJPEfKFw?si=NlhQ4hseQ2OEn_wQKlq2Vg",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evNZY5NHq?si=459ea47c15ed41fc",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO12tsHe?si=0849edf5c7744be7",

"https://open.spotify.com/album/6dVIqQ8qmQ5GBnJ9shOYGE?si=w_sR1CzYQ1y9zTSg6MG6VA",

"https://open.spotify.com/album/3gBVdu4a1MMJVMy6vwPEb8?si=QqmH_IosSEOOmCibsI3UAg",

"https://open.spotify.com/playlist/37i9dQZF1DXbTxeAdrVG2l?si=502380d7b2ac470e",

"https://open.spotify.com/playlist/37i9dQZF1DX7iB3RCnBnN4?si=7d8b96f054ff4d0f",

"https://open.spotify.com/album/0EiI8ylL0FmWWpgHVTsZjZ?si=ObKQ_eL4SXCxBLtmgkhFrg",

"https://open.spotify.com/album/5QG3tjE5L9F6O2vCAPph38?si=z72OCqp4SYS4-vR-TG4Flg",

"https://open.spotify.com/playlist/37i9dQZF1DX4o1oenSJRJd?si=ea8dcd40cef2457d",

"https://open.spotify.com/album/3vOgbDjgsZBAPwV2M3bNOj?si=6EK7j-x3Q_apbJZ9OKXHYA",

"https://open.spotify.com/album/7fRrTyKvE4Skh93v97gtcU?si=aznYBNYMSKKR1usyTC-o8w",

"https://open.spotify.com/album/3OxfaVgvTxUTy7276t7SPU?si=5vtDgLyhT8WqoDtAwYJFdw",

"https://open.spotify.com/album/7xV2TzoaVc0ycW7fwBwAml?si=tVYfvGNaRFWGzByq1rFcnQ",

"https://open.spotify.com/album/5r36AJ6VOJtp00oxSkBZ5h?si=fBesM3ecShaYncJ3mi7DRA",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO0yAm6Q?si=13ff89d8dd4b467d",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO1T5zSU?si=1eb81a6a19974c96",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO06Ki7m?si=0eaabcaa647848f6",

"https://open.spotify.com/album/41GuZcammIkupMPKH2OJ6I?si=fozaEeGAS6-MvzMug6DC8g",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3x2k4o?si=9717236426ba41c1",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO4gtw7S?si=496d913e3e1d48ae",

"https://open.spotify.com/playlist/2WxbQjSs5xcKRRcgIH5xQW?si=6c2b8e54ef514eb3",

"https://open.spotify.com/playlist/37i9dQZF1DXbrUpGvoi3TS?si=348ddf4129bb4467"

"https://open.spotify.com/album/6s84u2TUpR3wdUv4NgKA2j?si=TuwMgQKvSKS2kAKKM1pCWA",

"https://open.spotify.com/album/1D06fz3cuob62ysTS8k6gu?si=cfyKpvd9Ry2NrPn_PP0Tvg",

"https://open.spotify.com/album/3euz4vS7ezKGnNSwgyvKcd?si=EbogJTvwTNecFPPFuM9S8Q",

"https://open.spotify.com/album/1NAmidJlEaVgA3MpcPFYGq?si=PIPkQ6jgT7Kpgn_s0U6_Sw",

"https://open.spotify.com/playlist/37i9dQZF1E4AfEUiirXPyP?si=16dd34da17f04323",

"https://open.spotify.com/playlist/37i9dQZF1DX50MDbDwt4w8?si=596b0fa5dfd948fe",

"https://open.spotify.com/album/1BaHo66NCQNx6ku0hPn9bR?si=QDlmk-oNRQCKpKJOW5BvIw",

"https://open.spotify.com/album/6KaEpgeZQF6ZVVAmSoZUrb?si=vJ3-FOUBSEm6rXWoaLVyPA",

"https://open.spotify.com/playlist/37i9dQZF1DXcBWIGoYBM5M?si=4268aa7dd8214c0a",

"https://open.spotify.com/album/0FzWvaeMBfKBCqxHTLVlB8?si=TM063eisT8qolyeLP5KmMw",

"https://open.spotify.com/album/6GjwtEZcfenmOf6l18N7T7?si=3qZYq2ESSa-IQcwNTJH-Vg",

"https://open.spotify.com/album/3RQQmkQEvNCY4prGKE6oc5?si=VFR5pJwvSwaBHRjfhaO-sA",

"https://open.spotify.com/album/4FftCsAcXXD1nFO9RFUNFO?si=TWksAVchRRax_pb7rFLLnA",

"https://open.spotify.com/album/5vD5M5VW62LL78Ko8x0CVZ?si=lDT1qQzeSpS-rkoWA9408A",

"https://open.spotify.com/album/5dGWwsZ9iB2Xc3UKR0gif2?si=BbSZMPsMSXW8ghI2XMeN-Q",

"https://open.spotify.com/playlist/37i9dQZF1DXaQm3ZVg9Z2X?si=79a12d5161ac4944",

"https://open.spotify.com/album/6ZG5lRT77aJ3btmArcykra?si=AhimBovFTwCKmUshStN_bg",

"https://open.spotify.com/album/3cfAM8b8KqJRoIzt3zLKqw?si=6JokMXr5Q1Gk4zn0r5tIqA",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3CRVnO?si=0f40054c8aea460d",

"https://open.spotify.com/album/2QRedhP5RmKJiJ1i8VgDGR?si=sbPAy8zOQ86Cbc-vChLUrA",

"https://open.spotify.com/album/7dAm8ShwJLFm9SaJ6Yc58O?si=sqy4ILtHRv-dWSGG1Q3qUQ",

"https://open.spotify.com/album/5MQBzs5YlZlE28mD9yUItn?si=cnqoJcY-RsyyF6Bk47K45g",

"https://open.spotify.com/playlist/37i9dQZF1DX08mhnhv6g9b?si=da1c5cc32b5f47a9",

"https://open.spotify.com/album/43wFM1HquliY3iwKWzPN4y?si=TqpbERcHR0-8876GfGlgaQ",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO0sQwP6?si=b3082bf6733a4ce6",

"https://open.spotify.com/album/4rG0MhkU6UojACJxkMHIXB?si=U9d0i4VeQ86J1uL5hG7Atg",

"https://open.spotify.com/album/76QqoE30i9HVwxtxYMkWXT?si=5630DbDcRUSEvoy88jgiWA",

"https://open.spotify.com/album/3sysiYphqNRQw7VKLCg1yE?si=HSwYbFk1R8a2K9vTdzhqWA",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO0jO79m?si=c16feef9e8284358",

"https://open.spotify.com/album/6bUxh58rYTL67FS8dyTKMN?si=V8XZ_5flR1K5pylhBU-bhw",

"https://open.spotify.com/album/2gOSZzsO8EZJvtdmVOVtJE?si=j4HZJ16PQuKg2nnu_M_-fA",

"https://open.spotify.com/album/40GMAhriYJRO1rsY4YdrZb?si=xyCs3hsTSLCMQ_uML8C0Wg",

"https://open.spotify.com/album/1ATL5GLyefJaxhQzSPVrLX?si=QzoWUWZqQqeoJqw6yzZRiw",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO1XGbvi?si=6733dc1161644367",

"https://open.spotify.com/album/4iqbFIdGOTzXeDtt9owjQn?si=gaBcXjO_Trmt6NTW7LWltg",

"https://open.spotify.com/playlist/4hMcqod7ERKJ9mtjgdimeV?si=ff81ce8f7dfa4288",

"https://open.spotify.com/playlist/64r1Ry0JIWHboowR4LWp5R?si=eb99ae0a6cc2427e",

"https://open.spotify.com/playlist/2SM6rniZl84fEyMCB5KMQB?si=c50ac3d4fcb2414c",

"https://open.spotify.com/playlist/3yiEIxeirwDdeSBPvA3ATA?si=75ed8bdd49024d23",

"https://open.spotify.com/album/5duyQokC4FMcWPYTV9Gpf9?si=ZqwkKwcWRVyNCmC6cwqfoA",

"https://open.spotify.com/album/1vWMw6pu3err6qqZzI3RhH?si=q5KutEKCSYOwkMMyv_CeLQ",

"https://open.spotify.com/album/3o5EnVZNJXtfPV8tCoagjI?si=VC1pUyceTt2W60B-NqW8tQ",

"https://open.spotify.com/album/4LH4d3cOWNNsVw41Gqt2kv?si=IVCrJFLhTVCtnrAbl7BPNA",

"https://open.spotify.com/album/0bCAjiUamIFqKJsekOYuRw?si=WdlRiUrLRNG_Y3nzfT8zVg",

"https://open.spotify.com/album/0ETFjACtuP2ADo6LFhL6HN?si=XtP-PgkWR3GqF7agZTGbDQ"

"https://open.spotify.com/album/1C2h7mLntPSeVYciMRTF4a?si=JLGQgQ0fSFOSFdib03Gmvw",

"https://open.spotify.com/album/3Us57CjssWnHjTUIXBuIeH?si=9TbzmYjITBKOKKd-W__c4Q",

"https://open.spotify.com/album/6ucy4v9cUETA0yRQx8D34F?si=AhdpZDO8TMG8NVgkmuIPiQ",

"https://open.spotify.com/album/6ucy4v9cUETA0yRQx8D34F?si=AhdpZDO8TMG8NVgkmuIPiQ",

"https://open.spotify.com/playlist/37i9dQZF1DX5Vy6DFOcx00?si=fc6c2c55c7e04fe8",

"https://open.spotify.com/album/5Oli3gQJrHdahY7FDEoofW?si=WtYM5IppSruFOxW9nznCmA",

"https://open.spotify.com/album/2ODvWsOgouMbaA5xf0RkJe?si=wJU41kxpSSaWQ3hi5oIz2g",

"https://open.spotify.com/playlist/6Wbz2Y8J5Q9lqvpAUCCsK9?si=5671b6a23c7a45f6",

"https://open.spotify.com/playlist/1iV7itOMvqBlGASzrrAc9R?si=01b8260763274b1d",

"https://open.spotify.com/album/1MPAXuTVL2Ej5x0JHiSPq8?si=ftlteFTNRV60FrEZPzRfhQ",

"https://open.spotify.com/playlist/7vCQya7MR49M93F9D8su99?si=fdaaee31afa34bb5",

"https://open.spotify.com/playlist/37i9dQZF1DX2vTOtsQ5Isl?si=04fe4154ca4843ef",

"https://open.spotify.com/playlist/6mtYuOxzl58vSGnEDtZ9uB?si=2b169ed60e224cd0",

"https://open.spotify.com/album/7HnbhIDKXIBhMR4EPGuMgu?si=q2jev-UoSfSO08gbl-C6Cw",

"https://open.spotify.com/album/69XPm6tYJKIXL7dvdDCPXM?si=2ace7wSGSza05d9XVRCmpw",

"https://open.spotify.com/album/3xl0OvcSlc9Mwe5ToaFtD3?si=UcHspJ4xTUeYjj5zxweskg",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3DbIU8?si=85cd9687052d4513",

"https://open.spotify.com/album/4JPguzRps3kuWDD5GS6oXr?si=lmyzn-HcSTiP-SIZ0l9L4A",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO15Ttp6?si=33b30f8b2a2c4fd8",

"https://open.spotify.com/album/69YLD3zIxQt55bHjVlhLTP?si=8ylSql68QWC9HgPvBzt6bQ",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO1rYkRG?si=c51b0f6c24464271",

"https://open.spotify.com/album/5BNUgsVK7qqfFnV046qHfW?si=KwXrkmOyTPyfyantE10L_g",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO44TgnC?si=8847d6589d634e73",

"https://open.spotify.com/album/57lgFncHBYu5E3igZnuCJK?si=DNw9iC5OROqZVEcmZ6ejpA",

"https://open.spotify.com/album/0bJIHF1Or1YBLFBMwv53K2?si=pLE2E8jJQTSgeqBY3ZlPSQ",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO4x3X2w?si=3e2b5f6aa6cc40c7",

"https://open.spotify.com/album/3XYqOJI1YlX40kJTdzFEzp?si=K9AO6uoWRoiLQBpIUirsUA",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO2piM6c?si=95a81b1994db4e94",

"https://open.spotify.com/album/3uSWaQxJAdm5MWKQkQJNoK?si=-YChd3OzSdqx7I0hk8O0kQ",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3M0Fbi?si=65a4f1e0e6af4995",

"https://open.spotify.com/album/3FFGbUutKWN1c4f0CJR4Uh?si=2OMERW48SR2WEziyvOvmXQ",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO1HeimY?si=3c10a62f25a4439c",

"https://open.spotify.com/album/3QutrFTKwcT0Wn99v3u9cw?si=2YnZ_d49TeitqYL6dcpIjA",

"https://open.spotify.com/album/5WxTKN6iN2qOySNMOsJrM6?si=PpILoc8XSa2ikUYOqGWlSQ",

"https://open.spotify.com/album/7iykGeMdxOdYtNRtYCNaMA?si=dhYtDV0TQ_WFv9h6sPNN9A",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO0o2b28?si=31d1bbf673ce462e",

"https://open.spotify.com/album/1EUOmPNn2Fw6dwsjxKLFmC?si=cVUl1zRgS66wyAtHRH8mHw",

"https://open.spotify.com/album/4EeqIKMCBghfKdAXVgsQBt?si=R-mTrwWkSi-gR1YGXP12kg",

"https://open.spotify.com/album/0GaffqvjUTs1g9vDLAKFeV?si=PAGroDMSQiaiJQVFsiAQVQ",

"https://open.spotify.com/album/4p50wCLmX3dorhUIDFIYF2?si=zTbFOXxzQ_GxHLwVdkYo_w",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO3jxPig?si=7f9af2c7ed6349fa",

"https://open.spotify.com/album/6udoWtucgo5nrmcLhRrFNR?si=S-LMHqLQTAamKR9L1hk0rg",

"https://open.spotify.com/album/5kV0KBXfELibs6qQJLmOtg?si=DwFCQP5ATQ-Jp7fMugbERA",

"https://open.spotify.com/album/2yXnY2NiaZk9QiJJittS81?si=bh8CJrkdR4iJ2SQFjP6Fng",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO1bSHqE?si=40ddc9e4fae64dc5",

"https://open.spotify.com/album/0z7pVBGOD7HCIB7S8eLkLI?si=m8gs_-T5TeyqQ7Y5K95EyQ",

"https://open.spotify.com/playlist/37i9dQZF1DZ06evO2yXXGB?si=71b575c9c0db464d",

"https://open.spotify.com/album/2vD3zSQr8hNlg0obNel4TE?si=yE2szlZXS727LSLWcITMbw",

"https://open.spotify.com/album/6cunQQ7YZisYOoiFu2ywIq?si=8NDLTTO_QoKbgdNx17gXpg",

"https://open.spotify.com/album/6FJxoadUE4JNVwWHghBwnb?si=ErK103sORsaSVsjnwbUudA",

"https://open.spotify.com/album/7dK54iZuOxXFarGhXwEXfF?si=xrbFYU2fTkuLtGNZtSGGNA",

"https://open.spotify.com/album/7mvXPtV4jvA1hp5Wx2FAJA?si=Ru3JCJUQQEiOUzsVBhCvLw",

"https://open.spotify.com/album/1BaHo66NCQNx6ku0hPn9bR?si=R15sGSlHSQunzax8yIlsVA",

"https://open.spotify.com/album/1ks1k7zln4TyBjn1CF3vCz?si=P4cphKbZQoOYF7L8Jd8bjQ",

"https://open.spotify.com/album/5LGsh3kexUfi3qkIIxb8vK?si=GKeh0JKXQgS4y7RRQTArog",

"https://open.spotify.com/album/1QRP5lutJodPixU2EWfnD7?si=APRM0QvyTmqzOqdxCrMXfg"

]

playlist_ids = []
album_ids = []

for link in links:
    if "playlist" in link:
        playlist_ids.append(link.split("/playlist/")[1].split("?")[0])
    elif "album" in link:
        album_ids.append(link.split("/album/")[1].split("?")[0])


print("Playlists:", playlist_ids[:3], "...")  # Show first 3 as example
print("Albums:", album_ids[:3], "...")

Playlists: ['37i9dQZF1DZ06evNZVVBPG', '37i9dQZF1E4sa1kOvRGgMb', '37i9dQZF1E4sa1kOvRGgMb'] ...
Albums: ['1pzvBxYgT6OVwJLtHkrdQK', '6DEjYFkNZh67HP7R9PSZvv', '5L5evi5tJPh8WaEFAQp7Tp'] ...


In [ ]:
def check_spotify_item(item_id, item_type):
    url = f"https://api.spotify.com/v1/{item_type}s/{item_id}"
    resp = requests.get(url, headers=headers, params={"market": "US"})
    if resp.status_code == 200:
        print(f"✅ {item_type.capitalize()} works: {item_id}")
        return True
    elif resp.status_code == 404:
        print(f"❌ {item_type.capitalize()} not found or private: {item_id}")
        return False
    else:
        print(f"⚠️ {item_type.capitalize()} error ({resp.status_code}): {item_id}")
        return False


playlist_ids 
album_ids 

valid_playlists = [pid for pid in playlist_ids if check_spotify_item(pid, "playlist")]
valid_albums = [aid for aid in album_ids if check_spotify_item(aid, "album")]

print("\n==================== RESULTS ====================")
print(f"🎵 Working playlists: {len(valid_playlists)} / {len(playlist_ids)}")
print(f"💿 Working albums: {len(valid_albums)} / {len(album_ids)}")
print("=================================================")

print("\n✅ List of working playlists:", valid_playlists)
print("✅ List of working albums:", valid_albums)

In [22]:
# we have total of 50 valid playlists and 116 valid albums to work with
# the rest is either private or removed
# now we will fetch tracks from these valid playlists and albums

def get_playlist_tracks(playlist_id):
    tracks = []
    url = f"https://api.spotify.com/v1/playlists/{playlist_id}/tracks"
    while url:
        resp = requests.get(url, headers=headers)
        data = resp.json()
        for item in data.get('items'):
            if item.get('track') and item['track'].get('id'):
                tracks.append(item['track']['id'])
        url = data.get('next')
        time.sleep(0.5)
    return tracks

def get_album_tracks(album_id):
    tracks = []
    url = f"https://api.spotify.com/v1/albums/{album_id}/tracks"
    while url:
        resp = requests.get(url, headers=headers)
        data = resp.json()
        for item in data['items']:
            if item.get('id'):
                tracks.append(item['id'])
        url = data.get('next')
        time.sleep(0.5)
    return tracks

all_track_ids = set()
for pid in valid_playlists:
    all_track_ids.update(get_playlist_tracks(pid))
for aid in valid_albums:
    all_track_ids.update(get_album_tracks(aid))

print(f"Total unique tracks: {len(all_track_ids)}")

Total unique tracks: 8774


In [ ]:
# we have over 8700 unique tracks from the valid playlists and albums
# now we will fetch detailed data for each track including artist and album info
def get_selected_track_data(track_id):
    """
    Fetch selected data for a track, its main artist, and album from Spotify API.
    Returns a dict or None if the call fails.
    """
    # Get track details
    track_url = f"https://api.spotify.com/v1/tracks/{track_id}"
    try:
        track_resp = requests.get(track_url, headers=headers, timeout=10)
    except Exception as e:
        print(f"Error fetching track {track_id}: {e}")
        return None
    if track_resp.status_code != 200:
        print(f"Track fetch failed {track_id}: {track_resp.status_code}")
        return None
    track = track_resp.json()

    # Get album details
    album = track.get('album', {})

    # Get main artist details
    artist = {}
    artist_id = track['artists'][0]['id'] if track.get('artists') else None
    if artist_id:
        artist_url = f"https://api.spotify.com/v1/artists/{artist_id}"
        try:
            artist_resp = requests.get(artist_url, headers=headers, timeout=10)
            if artist_resp.status_code == 200:
                artist = artist_resp.json()
            else:
                print(f"Artist fetch failed {artist_id}: {artist_resp.status_code}")
        except Exception as e:
            print(f"Error fetching artist {artist_id}: {e}")

    return {
        "track_id": track.get("id"),
        "track_name": track.get("name"),
        "track_number": track.get("track_number"),
        "track_popularity": track.get("popularity"),
        "track_duration_ms": track.get("duration_ms"),
        "explicit": track.get("explicit"),
        "artist_name": artist.get("name"),
        "artist_popularity": artist.get("popularity"),
        "artist_followers": (artist.get("followers", {}) or {}).get("total"),
        "artist_genres": artist.get("genres"),
        "album_id": album.get("id"),
        "album_name": album.get("name"),
        "album_release_date": album.get("release_date"),
        "album_total_tracks": album.get("total_tracks"),
        "album_type": album.get("album_type"),
    }


progress_file = "track_data_progress.csv"
backup_file = progress_file + ".bak"

# --- Resume from last progress if exists ---
if os.path.exists(progress_file):
    df_progress = pd.read_csv(progress_file)
    # Normalize to str for safe comparison
    processed_ids = set(df_progress.get('track_id', pd.Series(dtype=str)).astype(str))
    track_data = df_progress.to_dict('records')
    print(f"Resuming from {len(processed_ids)} tracks...")
else:
    processed_ids = set()
    track_data = []
    print("Starting fresh...")

save_every = 10  # Save every 10 newly added tracks
new_since_save = 0

# --- Main Loop ---
for i, tid in enumerate(all_track_ids):
    tid = str(tid)  # Normalize ID type

    if tid in processed_ids:
        continue  # Skip already processed

    print(f"Processing track {len(processed_ids)+1}: {tid}")
    info = get_selected_track_data(tid)
    if info:
        track_data.append(info)
        processed_ids.add(tid)
        new_since_save += 1

    
    if new_since_save >= save_every:
        print(f"Processed {len(processed_ids)} tracks total, saving progress...")
        temp_file = progress_file + ".tmp"
        # Atomic write to prevent corruption
        pd.DataFrame(track_data).to_csv(temp_file, index=False)
        os.replace(temp_file, progress_file)
        # Best-effort backup copy
        try:
            shutil.copy(progress_file, backup_file)
        except Exception as e:
            print(f"Backup warning: {e}")
        new_since_save = 0

    time.sleep(2)  # Avoid hitting rate limits

# --- Final save ---
print("Final save in progress...")
pd.DataFrame(track_data).to_csv(progress_file, index=False)
try:
    shutil.copy(progress_file, backup_file)
except Exception as e:
    print(f"Backup warning: {e}")

# Also save as final file
df_tracks = pd.DataFrame(track_data)
df_tracks.to_csv("track_data_final.csv", index=False)

print(f"\n✅ Done! Total tracks saved: {len(processed_ids)}")
print(df_tracks.head())

In [25]:
df_tracks.head(10)

,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,213173,False,Britney Spears,80.0,17755451.0,['pop'],325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,58,compilation
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,158463,True,Billie Eilish,90.0,118692183.0,[],0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,16,album
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57,267013,False,Rihanna,90.0,68997177.0,[],5QG3tjE5L9F6O2vCAPph38,Loud,2010-01-01,13,album
5,7H0ya83CMmgFcOhw0UB6ow,Space Song,3,77,320466,False,Beach House,72.0,2803036.0,['dream pop'],194CqC2Zi0kUFEPWedb3qr,Depression Cherry,2015-08-28,9,album
6,41zXlQxzTi6cGAjpOXyLYH,idontwannabeyouanymore,2,78,203569,False,Billie Eilish,90.0,118692183.0,[],7fRrTyKvE4Skh93v97gtcU,dont smile at me,2017-12-22,9,single
7,13jRFAGT8qd6aBwtJySlUm,Allein Allein - BENNETT Remix,1,52,145977,False,Alok,76.0,11247155.0,"['brazilian bass', 'electronic', 'slap house',...",1WKoDELzbFRR6UWNGh50LO,Allein Allein (feat. FREY) [BENNETT Remix],2024-09-27,2,single
8,0N5zjRnf8AreOm95iSBXF4,Even My Dad Does Sometimes,15,50,228533,False,Ed Sheeran,88.0,122773292.0,['soft pop'],1xn54DMo2qIqBuMqHtUsFd,x (Deluxe Edition),2014-06-21,16,album
9,0qUcpOOna3kkrwfqky85e1,Eyes Blue Like The Atlantic (feat. Subvrbs),1,63,154599,False,Sista Prod,48.0,68226.0,[],5UYjxc4HIYeesKS0WJlhEI,Eyes Blue Like The Atlantic (feat. Subvrbs),2020-07-20,1,single


In [38]:
df_tracks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8778 entries, 0 to 8777
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   track_id            8778 non-null   object 
 1   track_name          8776 non-null   object 
 2   track_number        8778 non-null   int64  
 3   track_popularity    8778 non-null   int64  
 4   track_duration_ms   8778 non-null   int64  
 5   explicit            8778 non-null   bool   
 6   artist_name         8774 non-null   object 
 7   artist_popularity   8774 non-null   float64
 8   artist_followers    8774 non-null   float64
 9   artist_genres       8774 non-null   object 
 10  album_id            8778 non-null   object 
 11  album_name          8776 non-null   object 
 12  album_release_date  8778 non-null   object 
 13  album_total_tracks  8778 non-null   int64  
 14  album_type          8778 non-null   object 
dtypes: bool(1), float64(2), int64(4), object(8)
memory usag

In [27]:
df_tracks.isnull().sum()

track_id              0
track_name            2
track_number          0
track_popularity      0
track_duration_ms     0
explicit              0
artist_name           4
artist_popularity     4
artist_followers      4
artist_genres         4
album_id              0
album_name            2
album_release_date    0
album_total_tracks    0
album_type            0
dtype: int64

In [37]:
df_tracks.drop(columns=["artist_genres"]).duplicated().sum()

np.int64(0)

# we have our data set ready with 8778 rows and we can now proceed with cleaning  